# 🧪 `pdb2reaction` — reaction-mechanism GUI (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/t-0hmura/pdb2reaction/blob/main/examples/pdb2reaction_colab.ipynb)

> **Each user runs this in their own Colab** (your Google account, your GPU) — nothing is shared. Use a **GPU** runtime (Runtime ▸ Change runtime type ▸ GPU). The first **Setup** run installs pdb2reaction and one MLIP backend.

Enzymatic reaction paths from PDB or mmCIF structures with ML interatomic potentials, driven from **one in-notebook GUI**: **Setup → Launch GUI**, then *Input → click residues/atoms in 3D to set `-c`/`-l`/`-s` → Options → Validate → Run*. Chain-qualified selections remain unambiguous for repeated residue names and large structures. The editable command line is always the exact command that Run executes.

> Backend defaults to **MACE** (no login). UMA is Hugging-Face-gated — switch in Setup after accepting its license. Repo: <https://github.com/t-0hmura/pdb2reaction>

In [ ]:
#@title  ⚙️ Setup — install pdb2reaction + a backend  (first run ~3-5 min) { display-mode: "form" }
#@markdown Only the **selected backend** is installed (MACE and UMA cannot coexist because their `e3nn` requirements conflict). To switch, restart the runtime and run Setup again.
#@markdown **MACE = no login; UMA = gated Hugging Face model.** The release ref defaults to the notebook's matching version.
backend = "mace"  #@param ["mace", "uma"]
pdb2reaction_ref = "v0.4.12"  #@param {type:"string"}
test_backend_model = False  #@param {type:"boolean"}
import os, sys, subprocess
if globals().get('BACKEND') not in (None, backend):
    raise RuntimeError('Backend switch requested. Restart the Colab runtime first, then rerun Setup.')
try: _gpu = subprocess.run(['nvidia-smi','-L'], capture_output=True, text=True).stdout.strip()
except FileNotFoundError: _gpu = ''   # nvidia-smi absent on CPU runtimes -> don't crash Setup
print(_gpu or '⚠️ No GPU detected — for real runs pick a GPU runtime (Runtime ▸ Change runtime type ▸ GPU). The GUI still loads.')
print('Installing the %s backend (a few minutes on first run)…' % backend)
def pip(*a): subprocess.run([sys.executable,'-m','pip','install','-q',*a], check=True)
REPO_DIR = 'pdb2reaction-src'
if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    if os.path.exists(REPO_DIR): raise RuntimeError('%s exists but is not a git clone; rename or remove it.' % REPO_DIR)
    subprocess.run(['git','clone','--filter=blob:none','--no-checkout',
                    'https://github.com/t-0hmura/pdb2reaction', REPO_DIR], check=True)
subprocess.run(['git','-C',REPO_DIR,'fetch','--depth','1','origin',pdb2reaction_ref], check=True)
subprocess.run(['git','-C',REPO_DIR,'checkout','--detach','FETCH_HEAD'], check=True)
pip('./' + REPO_DIR)
if backend == 'mace':
    subprocess.run([sys.executable,'-m','pip','uninstall','-y','-q','fairchem-core'], check=False)
    pip('mace-torch>=0.3.8'); print('MACE installed (no login needed).')
else:
    # UMA is gated: accept the FAIR Chemistry License at https://huggingface.co/facebook/UMA
    from huggingface_hub import login, notebook_login
    if os.environ.get('HF_TOKEN'): login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    else: notebook_login()
    print('UMA installed (uses fairchem-core; needs HF license + login).')
pip('py3Dmol','ipywidgets','matplotlib','nglview')
BACKEND = backend; TOOL = 'pdb2reaction'
from importlib.metadata import version
commit = subprocess.check_output(['git','-C',REPO_DIR,'rev-parse','--short','HEAD'], text=True).strip()
installed_version = version('pdb2reaction')
if pdb2reaction_ref.startswith('v') and installed_version != pdb2reaction_ref[1:]:
    raise RuntimeError('Installed version %s does not match requested ref %s.' % (installed_version, pdb2reaction_ref))
print('pdb2reaction', installed_version, '| ref', pdb2reaction_ref, '| commit', commit)
if test_backend_model:
    from pdb2reaction.backends import create_calculator
    model = 'MACE-OMOL-0' if backend == 'mace' else 'uma-s-1p2'
    create_calculator(backend=backend, model=model, charge=0, spin=1)
    print('Backend model load OK:', model)
print('\n✅ Setup done. backend =', BACKEND, '— now run "Launch GUI".')

In [ ]:
#@title Launch GUI  (run once, after Setup) { display-mode: "form" }
import os, glob, json, shlex, math, shutil, subprocess
from pathlib import Path
import ipywidgets as W
from IPython.display import display, clear_output, Image, HTML
import py3Dmol
try: TOOL
except NameError: TOOL = 'pdb2reaction'
try: BACKEND
except NameError: BACKEND = 'mace'
try: REPO_DIR
except NameError: REPO_DIR = 'pdb2reaction-src'
IS_CLUSTER = True
try:
    import nglview as nv
    try:
        from google.colab import output as _cwm; _cwm.enable_custom_widget_manager()
    except Exception:
        _cwm = None        # custom-widget-manager is Colab-only; skip elsewhere
    _HAVE_NGL = True
except Exception:
    _HAVE_NGL = False       # NGLView is an optional rich viewer; py3Dmol is primary

ACCENT = '#114b8a'
display(HTML("""<style>
.rxapp { width:100%; max-width:100%; box-sizing:border-box; }
.rxapp, .rxapp .widget-label, .rxapp .widget-html-content, .rxapp .widget-readout {
  font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Roboto,Helvetica,Arial,sans-serif !important;
  color:#1f2937; }
.rxapp .widget-html-content code { background:#f1f5f9; padding:1px 5px; border-radius:5px;
  font-family:ui-monospace,SFMono-Regular,Menlo,monospace; font-size:12px; color:#334155;
  overflow-wrap:anywhere; word-break:break-word; white-space:normal; }
.rxapp .widget-button { border-radius:9px !important; font-weight:500 !important;
  box-shadow:none !important; min-height:34px; }
.rxapp button.jupyter-button, .rxapp .widget-upload > button { min-height:34px !important; }
.rxapp .widget-dropdown select, .rxapp .widget-text input, .rxapp .widget-textarea textarea,
.rxapp input[type="number"] { border-radius:9px !important; border:1px solid #e2e8f0 !important; }
.rxapp .widget-button:focus-visible, .rxapp select:focus-visible,
.rxapp input:focus-visible, .rxapp textarea:focus-visible {
  outline:3px solid rgba(37,99,235,.28) !important; outline-offset:2px !important; }
.rxapp .widget-tab > .rxapp-tabs, .rxapp .p-TabBar-tab { font-weight:500; }
.rxcard { border:1px solid #e8ecf2; border-radius:14px; padding:12px 14px; background:#ffffff;
  box-shadow:0 1px 2px rgba(16,24,40,0.05); box-sizing:border-box; flex:1 1 250px; min-width:230px; }
.rxworkspace { align-items:stretch !important; column-gap:14px; row-gap:12px !important; }
.rxviewer { flex:2 1 520px !important; min-width:300px; }
.rxinspector { flex:1 1 300px !important; min-width:260px; }
.rxviewer .jupyter-widgets-output-area, .rxviewer iframe { width:100% !important; max-width:100% !important; }
.rxapp img, .rxapp svg, .rxapp iframe { max-width:100%; }
.rxcmd textarea { font-family:ui-monospace,SFMono-Regular,Menlo,monospace !important;
  font-size:13px !important; background:#0f172a !important; color:#7ee787 !important;
  border-radius:11px !important; line-height:1.5 !important; border:1px solid #1e293b !important;
  padding:9px 11px !important; }
.rxdrop { border:2px dashed #cbd5e1; border-radius:14px; padding:16px; background:#f8fafc;
  transition:all .15s ease; }
.rxdrop:hover { border-color:#94a3b8; background:#f1f5f9; }
.rxchip button { border-radius:999px !important; font-size:12px !important; padding:1px 10px !important; }
.rxrun button { font-weight:600 !important; }
.rxapp .widget-hbox { flex-wrap:wrap !important; row-gap:6px; }
.rxapp hr { border:none; border-top:1px solid #eef0f4; }
@media (max-width: 600px) {
  .rxapp .widget-dropdown, .rxapp .widget-text,
  .rxapp .widget-select-multiple { max-width:100% !important; }
  .rxapp .lm-TabBar-tab, .rxapp .p-TabBar-tab { min-width:0 !important; }
  .rxapp .lm-TabBar-tabLabel, .rxapp .p-TabBar-tabLabel {
    overflow:hidden; text-overflow:ellipsis; }
}
@media (prefers-reduced-motion: reduce) {
  .rxapp * { transition:none !important; animation:none !important; }
}
</style>"""))

CLI = 'pdb2reaction'
S = {'tool': TOOL, 'backend': BACKEND, 'model': ('uma-s-1p2' if BACKEND == 'uma' else 'MACE-OMOL-0'),
     'mode': None, 'inputs': [], 'subcmd': 'all',
     'parm': None, 'center': [], 'center_ids': [], 'lcharge': {},
     'scan_atoms': [None, None], 'scan_target': 1.6, 'scan_preset': '', 'scan_stages': [], 'scan_axes': [],
     'freeze_buf': [None, None], 'freeze_pairs': [], 'freeze_atoms': [], 'charge': 0,
     'charge_explicit': False,
     'tsopt': False, 'thermo': False, 'out_dir': 'result',
     '_last_out_dir': None,
     '_pdb_path': None, '_pdb_text': '', '_hetero': [], '_atoms': {}, '_atom_meta': [],
     '_installed_backend': BACKEND,
     'show_water': False, 'surface': False, 'spin': False,
     'measure_atoms': [], 'rep': 'cartoon', 'color': 'spectrum'}

_AA = {'ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE','LEU','LYS','MET',
       'PHE','PRO','SER','THR','TRP','TYR','VAL','HID','HIE','HIP','CYX','ASH','GLH',
       'LYN','SEC','PYL','MSE'}
_WATER = {'HOH','WAT','H2O','DOD','TIP','TIP3','SOL','T3P'}

def _pdb_coordinate_rows(text):
    rows = []
    for line in text.splitlines():
        if line[0:6].strip() not in ('ATOM', 'HETATM'): continue
        try:
            rows.append((float(line[30:38]), float(line[38:46]), float(line[46:54])))
        except ValueError:
            raise ValueError('Viewer bridge contains an invalid PDB coordinate row.')
    return rows

def parse_residues(text, metadata=None):
    allr, het = set(), set()
    if metadata:
        for atom in metadata:
            r = str(atom.get('resname') or '').strip().upper()
            if not r: continue
            allr.add(r)
            if r not in _WATER and r not in _AA: het.add(r)
        return sorted(allr), sorted(het)
    for ln in text.splitlines():
        if ln[0:6].strip() not in ('ATOM', 'HETATM'): continue
        r = ln[17:20].strip(); allr.add(r)
        if r not in _WATER and r not in _AA: het.add(r)
    return sorted(allr), sorted(het)

def parse_atoms(text, metadata=None):
    if metadata:
        coords = _pdb_coordinate_rows(text)
        if len(coords) != len(metadata):
            raise ValueError('Viewer PDB atom count does not match retained structure metadata.')
        out = {}
        for index, (meta, xyz) in enumerate(zip(metadata, coords)):
            meta['xyz'] = xyz
            meta['index'] = index
            key = (str(meta.get('chain') or ''), str(meta.get('resname') or ''),
                   str(meta.get('resseq')), str(meta.get('name') or ''))
            out[key] = xyz
        return out
    d = {}
    for ln in text.splitlines():
        if ln[0:6].strip() in ('ATOM', 'HETATM'):
            try:
                d[(ln[21:22].strip(), ln[17:20].strip(), ln[22:26].strip(), ln[12:16].strip())] = \
                    (float(ln[30:38]), float(ln[38:46]), float(ln[46:54]))
            except ValueError:
                continue
    return d

def _load_view_structure(path):
    """Return a safe viewer PDB plus original/auth atom metadata."""
    from pdb2reaction.core.utils import prepare_input_structure, load_pdb_atom_metadata
    with prepare_input_structure(Path(path)) as prepared:
        source = Path(prepared.geom_path)
        text = source.read_text(encoding='utf-8', errors='replace')
        metadata = load_pdb_atom_metadata(source)
        viewer_path = Path('pdb2reaction_gui_viewer_input.pdb')
        viewer_path.write_text(text, encoding='utf-8')
    parse_atoms(text, metadata)       # attach stable 0-based index + coordinates
    return text, metadata, str(viewer_path)

def _aspec(x):
    chain = str(x.get('chain') or '').strip()
    if chain:
        return '%s:%s:%s:%s' % (chain, x['resn'], x['resi'], x['atom'])
    return '%s %s %s' % (x['resn'], x['resi'], x['atom'])

def scan_literals():
    """One literal per stage (multiple = staged); tuples within a stage = concerted."""
    if S['scan_stages']:
        return ['[' + ','.join('(\"%s\",\"%s\",%g)' % (_aspec(bd['a']), _aspec(bd['b']), bd['t'])
                               for bd in stage) + ']' for stage in S['scan_stages']]
    if S['scan_preset'] and not all(S['scan_atoms']): return [S['scan_preset']]
    a, b = S['scan_atoms']
    if a and b: return ['[(\"%s\",\"%s\",%g)]' % (_aspec(a), _aspec(b), S['scan_target'])]
    return []

def scan2d_literal():
    """scan2d/scan3d need ONE -s with per-axis quadruples: [(a,b,low,high),...]."""
    ax = S.get('scan_axes', [])
    if not ax: return ''
    return '[' + ','.join('(\"%s\",\"%s\",%g,%g)' % (_aspec(a['a']), _aspec(a['b']), a['lo'], a['hi'])
                          for a in ax) + ']'

def freeze_pair_lit():
    if not S['freeze_pairs']: return ''
    parts = []
    for p in S['freeze_pairs']:
        if p.get('t') is not None:
            parts.append('(\"%s\",\"%s\",%g)' % (_aspec(p['a']), _aspec(p['b']), p['t']))
        else:
            parts.append('(\"%s\",\"%s\")' % (_aspec(p['a']), _aspec(p['b'])))
    return '[' + ','.join(parts) + ']'

def scan_distance():
    a, b = S['scan_atoms']
    if a and b and a.get('xyz') and b.get('xyz'):
        return math.dist(a['xyz'], b['xyz'])
    return None

def _xyz(d):
    if not d: return None
    if d.get('xyz') is not None: return d['xyz']
    return S['_atoms'].get((str(d.get('chain') or ''), d['resn'], str(d['resi']), d['atom']))

def _angle(a, b, c):                                   # angle at vertex b (degrees)
    u = [a[i] - b[i] for i in range(3)]; v = [c[i] - b[i] for i in range(3)]
    du = math.sqrt(sum(x * x for x in u)); dv = math.sqrt(sum(x * x for x in v))
    if not du or not dv: return None
    cos = max(-1.0, min(1.0, sum(u[i] * v[i] for i in range(3)) / (du * dv)))
    return math.degrees(math.acos(cos))

def _dihedral(p0, p1, p2, p3):                          # signed dihedral (degrees)
    b0 = [p0[i] - p1[i] for i in range(3)]; b1 = [p2[i] - p1[i] for i in range(3)]
    b2 = [p3[i] - p2[i] for i in range(3)]
    def cross(a, b): return [a[1]*b[2]-a[2]*b[1], a[2]*b[0]-a[0]*b[2], a[0]*b[1]-a[1]*b[0]]
    def dot(a, b): return sum(a[i] * b[i] for i in range(3))
    n = math.sqrt(dot(b1, b1)) or 1.0
    b1n = [x / n for x in b1]
    v = [b0[i] - dot(b0, b1n) * b1n[i] for i in range(3)]
    w = [b2[i] - dot(b2, b1n) * b1n[i] for i in range(3)]
    return math.degrees(math.atan2(dot(cross(b1n, v), w), dot(v, w)))

CLUSTER_SUBS = ['all', 'opt', 'sp', 'tsopt', 'freq', 'irc', 'dft', 'scan', 'scan2d', 'scan3d',
            'path-opt', 'path-search', 'extract', 'fix-altloc', 'add-elem-info',
            'energy-diagram', 'bond-summary', 'trj2fig']
MLMM_SUBS = ['all', 'opt', 'sp', 'tsopt', 'freq', 'irc', 'dft', 'scan', 'scan2d', 'scan3d',
             'path-opt', 'path-search', 'extract', 'define-layer', 'mm-parm', 'oniom-export',
             'oniom-import', 'fix-altloc', 'add-elem-info', 'energy-diagram', 'bond-summary', 'trj2fig']
SUBS = CLUSTER_SUBS if IS_CLUSTER else MLMM_SUBS
COMPUTE = {'all', 'opt', 'sp', 'tsopt', 'freq', 'irc', 'dft', 'scan', 'scan2d', 'scan3d',
           'path-opt', 'path-search'}
MLIP_COMPUTE = COMPUTE - {'dft'}
# Subcommands a non-advanced user can run entirely from the GUI (no CLI typing):
BASIC_SUBS = ['all', 'opt', 'sp', 'tsopt', 'freq', 'irc', 'dft', 'scan', 'path-opt', 'path-search']
SUBREQ = {
    'all': 'R + P structures (MEP) · or 1 structure + scan bond · or 1 TS + TS-only mode',
    'opt': 'one structure to optimize', 'sp': 'one structure (single-point E/F)',
    'tsopt': 'one TS-candidate structure', 'freq': 'one optimized structure',
    'irc': 'one TS structure', 'dft': 'one structure (DFT single-point)',
    'scan': 'one structure + a scan bond (pick atoms A & B in the Select tab)',
    'scan2d': 'one structure + 2 scan axes (pick bonds + low/high in the Select tab)',
    'scan3d': 'one structure + 3 scan axes (pick bonds + low/high in the Select tab)',
    'path-opt': 'two structures (segment endpoints)', 'path-search': 'two or more structures',
    'extract': 'a complex PDB/mmCIF + center residues (-c)',
}
# Keep the basic GUI chemically scoped: bare MACE "small/medium/large" aliases are
# materials checkpoints, not MACE-OMOL-0 variants. Advanced users can still type a
# different --backend-model in the editable command line.
MODELS = {'mace': ['MACE-OMOL-0'],
          'uma': ['uma-s-1p2', 'uma-s-1p1', 'uma-m-1p1']}
DEFAULT_MODEL = {'mace': 'MACE-OMOL-0', 'uma': 'uma-s-1p2'}

def _wv(name, default=None):
    """Safely read a widget's .value by global name (widget may not exist yet)."""
    w = globals().get(name)
    return getattr(w, 'value', default) if w is not None else default

def set_subcmd(v):
    S['subcmd'] = v
    dd = globals().get('dd_subcmd')
    if dd is not None and dd.value != v: dd.value = v

def build_cmd():
    if center_widget is not None: S['center'] = list(center_widget.value)
    if charge_rows is not None:
        S['lcharge'] = {r: x['val'].value for r, x in charge_rows.items() if x['use'].value}
    if not S['inputs']: raise ValueError('No input chosen (Input tab).')
    sub = _wv('dd_subcmd', S['subcmd']) or 'all'
    S['subcmd'] = sub
    bk = S.get('backend', BACKEND)
    if sub in MLIP_COMPUTE and bk != S.get('_installed_backend'):
        raise ValueError('Backend %s is not installed in this runtime; rerun Setup with backend=%s.' % (bk, bk))
    if sub == 'path-search' and len(S['inputs']) < 2:
        raise ValueError('path-search needs two or more structures in reaction order.')
    if sub == 'path-opt' and len(S['inputs']) != 2:
        raise ValueError('path-opt needs exactly two endpoint structures.')
    if sub == 'all':
        all_kind = _wv('all_mode', 'mep')
        if all_kind == 'mep' and len(S['inputs']) < 2:
            raise ValueError('all MEP mode needs two or more structures.')
        if all_kind == 'scan' and (len(S['inputs']) != 1 or not scan_literals()):
            raise ValueError('all scan mode needs one structure and a picked scan bond.')
        if all_kind == 'tsonly' and len(S['inputs']) != 1:
            raise ValueError('all TS-only mode needs exactly one TS candidate.')
    if sub == 'scan' and not scan_literals():
        raise ValueError('scan needs a picked atom pair and target distance.')
    if sub in ('scan2d', 'scan3d'):
        expected = 2 if sub == 'scan2d' else 3
        if len(S.get('scan_axes', [])) != expected:
            raise ValueError('%s needs exactly %d scan axes.' % (sub, expected))
    if not IS_CLUSTER and sub in COMPUTE and not S.get('parm'):
        raise ValueError('ML/MM compute commands require a matching Amber parm7 upload in Colab.')
    center_all = list(S['center']) + list(S.get('center_ids', []))   # resnames + residue-IDs
    lc = ','.join('%s:%g' % (k, v) for k, v in S['lcharge'].items())
    extra = shlex.split(_wv('adv_extra', '') or '')
    if sub not in COMPUTE and sub != 'extract':        # bespoke utility -> finish in the command line
        return [CLI, sub] + extra
    cmd = [CLI, sub, '-i', *S['inputs']]
    if sub in MLIP_COMPUTE: cmd += ['-b', bk]
    mdl = S.get('model')
    if sub in MLIP_COMPUTE and mdl and mdl != DEFAULT_MODEL.get(bk):
        cmd += ['--backend-model', mdl]
    output_arg = S['out_dir']
    if sub == 'extract' and Path(output_arg).suffix.lower() not in ('.pdb', '.ent', '.cif', '.mmcif'):
        output_arg = os.path.join(output_arg, 'cluster.pdb')
    cmd += ['-o', output_arg]
    if sub in ('all', 'extract') and center_all: cmd += ['-c', ','.join(center_all)]
    if lc: cmd += ['-l', lc]
    if not IS_CLUSTER and sub in COMPUTE: cmd += ['--parm', S['parm']]
    if sub in ('scan2d', 'scan3d'):
        if scan2d_literal(): cmd += ['-s', scan2d_literal()]  # one -s, per-axis (i,j,low,high) quadruples
    elif (sub == 'scan') or (sub == 'all' and len(S['inputs']) == 1):
        for lit in scan_literals(): cmd += ['-s', lit]        # multiple -s = staged stages
    gjf_supplies_charge = bool(S['inputs']) and all(Path(path).suffix.lower() == '.gjf' for path in S['inputs'])
    if (sub in COMPUTE and not lc and
            (sub != 'all' or S['mode'] != 'pdb' or not center_all) and
            (not gjf_supplies_charge or S.get('charge_explicit'))):
        cmd += ['-q', str(S['charge'])]
    m = _wv('adv_mult', 1)
    if sub in COMPUTE and m not in (None, 1): cmd += ['-m', str(int(m))]
    precision = _wv('adv_prec', 'auto')
    if sub in MLIP_COMPUTE and precision in ('fp32', 'fp64'): cmd += ['--precision', precision]
    if sub in MLIP_COMPUTE and _wv('adv_det', False): cmd += ['--deterministic']
    r = _wv('adv_radius', 0.0)
    if sub in ('all', 'extract', 'scan') and r and r > 0: cmd += ['-r', str(r)]
    th = _wv('adv_thresh', '(default)')
    if sub in ('all', 'opt', 'tsopt', 'path-opt') and th and th != '(default)': cmd += ['--thresh', th]
    if sub == 'opt' and freeze_pair_lit(): cmd += ['--dist-freeze', freeze_pair_lit()]
    if sub in ('opt', 'tsopt', 'path-opt', 'freq', 'irc') and S['freeze_atoms']:
        cmd += ['--freeze-atoms', ','.join(str(i) for i in S['freeze_atoms'])]
    if sub == 'all':
        if S['tsopt']: cmd.append('--tsopt')
        if S['thermo']: cmd.append('--thermo')
        mep = _wv('adv_mep', '(default)')
        if mep and mep != '(default)': cmd += ['--mep-mode', mep]
        if _wv('adv_dft', False):
            cmd.append('--dft')
            fb = _wv('adv_dftfb', '')
            if fb: cmd += ['--dft-func-basis', fb]
    cmd += extra
    return cmd

# -------- PyMOL-style editable command line + readiness chip ------------------
ready_chip = W.HTML()
toast = W.HTML()
run_status = W.HTML()
cmd_box = W.Textarea(value='', placeholder='the command builds here — edit it freely, then Run (like a PyMOL command line)',
                     layout=W.Layout(width='99%', height='66px'))
cmd_box.add_class('rxcmd')
_OK = '<span style="background:#1f7a3d;color:#fff;padding:2px 9px;border-radius:11px;font-size:12px;">● ready to run</span>'
_NO = '<span style="background:#9aa0a6;color:#fff;padding:2px 9px;border-radius:11px;font-size:12px;">○ %s</span>'
_auto = {'on': True, 'guard': False}
def _set_cmd(txt):
    _auto['guard'] = True; cmd_box.value = txt; _auto['guard'] = False
def refresh(_=None):
    try:
        built = build_cmd()
        if _auto['on']: _set_cmd(' '.join(shlex.quote(c) for c in built))
        ready_chip.value = _OK if _auto['on'] else _OK.replace('ready to run', 'ready (manual edit)')
    except Exception as e:
        if _auto['on']: _set_cmd('# ' + str(e))
        ready_chip.value = _NO % e
    _rs = globals().get('_render_summary')
    if _rs is not None: _rs()
    _rc = globals().get('_render_chips')
    if _rc is not None: _rc()
    _ro = globals().get('_render_output_note')
    if _ro is not None: _ro()
def _on_cmd_edit(_):
    if not _auto['guard']: _auto['on'] = False   # user took manual control of the line
cmd_box.observe(_on_cmd_edit, names='value')

# ============================================================== INPUT tab
input_msg = W.HTML()
def _goto(i): app.selected_index = i

def load_pdb(paths, parm=None, center=None, lcharge=None, scan_preset='', mode='pdb'):
    paths = [p for p in paths if p]
    S.update(inputs=paths, parm=parm, mode=mode, subcmd='all',
             center=list(center or []), center_ids=[], lcharge=dict(lcharge or {}),
             scan_atoms=[None, None], scan_preset=scan_preset)
    set_subcmd('all')
    am = globals().get('all_mode')
    if am is not None:
        am.value = 'mep' if len(paths) >= 2 else ('scan' if scan_preset else am.value)
    build_selection(); _goto(1); refresh()

_acc = '.pdb,.ent,.cif,.mmcif,.xyz,.gjf' if IS_CLUSTER else '.pdb,.ent,.cif,.mmcif,.parm7'
upl = W.FileUpload(accept=_acc, multiple=True, description='Drop files', icon='upload',
                   layout=W.Layout(width='260px'))
def _on_upload(change):
    items = upl.value
    pairs = items.items() if isinstance(items, dict) else [(f['name'], f) for f in items]
    structures, smalls, parm = [], [], None
    for name, meta in pairs:
        with open(name, 'wb') as fh: fh.write(bytes(meta['content']))
        low = name.lower()
        if low.endswith(('.pdb', '.ent', '.cif', '.mmcif')): structures.append(name)
        elif low.endswith('.parm7'): parm = name
        else: smalls.append(name)
    if structures:
        input_msg.value = '✅ <b>%s</b>%s' % (', '.join(structures), ' + parm7 (no AmberTools)' if parm else '')
        load_pdb(structures, parm)
    elif parm:
        S['parm'] = parm
        loaded = ', '.join(S.get('inputs', [])) or '(load a PDB/mmCIF next)'
        input_msg.value = '✅ <b>%s</b> + <b>%s</b>' % (loaded, parm)
        refresh()
    elif smalls:
        S.update(mode='small', inputs=smalls,
                 charge_explicit=not all(Path(path).suffix.lower() == '.gjf' for path in smalls))
        set_subcmd('opt' if len(smalls) == 1 else 'all')
        input_msg.value = '✅ <b>%s</b> — set total charge in Options.' % ', '.join(smalls); refresh()
upl.observe(_on_upload, names='value')

_EX = (['BezA methyltransferase - cluster MEP (R->P)', 'HCN -> HNC - small molecule (fast)']
       if IS_CLUSTER else ['methyltransferase complex - scan mode'])
ex_choice = W.Dropdown(options=_EX, value=_EX[0], description='example',
                       style={'description_width': 'initial'},
                       layout=W.Layout(width='400px', max_width='100%'))
ex_btn = W.Button(description='Load example', icon='flask', layout=W.Layout(width='150px'))
def _load_example(_):
    sel = ex_choice.value
    if IS_CLUSTER and sel.startswith('HCN'):
        with open('hcn_r.xyz', 'w') as fh: fh.write('3\nHCN\nC 0 0 0\nN 0 0 1.16\nH 0 0 -1.07\n')
        with open('hcn_p.xyz', 'w') as fh: fh.write('3\nHNC\nC 0 0 0\nN 0 0 1.17\nH 0 0.98 1.60\n')
        S.update(mode='small', inputs=['hcn_r.xyz', 'hcn_p.xyz'], parm=None, center=[], center_ids=[],
                 lcharge={}, scan_atoms=[None, None], scan_stages=[], scan_preset='', charge=0,
                 charge_explicit=True)
        set_subcmd('all')
        wq = globals().get('w_q')
        if wq is not None: wq.value = 0
        input_msg.value = 'HCN -> HNC small-molecule MEP (gas-phase, -q 0) - go to Run.'
        refresh(); return
    if IS_CLUSTER:
        ex = os.path.join(REPO_DIR, 'examples')
        if not os.path.exists(os.path.join(ex, '1.R.pdb')):
            input_msg.value = '⚠️ Run Setup first.'; return
        input_msg.value = '⭐ BezA methyltransferase (R→P MEP).'
        load_pdb([os.path.join(ex, '1.R.pdb'), os.path.join(ex, '3.P.pdb')],
                 center=['SAM', 'GPP', 'MG'], lcharge={'SAM': 1, 'GPP': -3})
    else:
        ex = os.path.join(REPO_DIR, 'examples', 'methyltransferase')
        if not os.path.exists(os.path.join(ex, 'complex.pdb')):
            input_msg.value = '⚠️ Run Setup first.'; return
        S['scan_target'] = 1.3
        input_msg.value = '⭐ Methyltransferase (scan mode, shipped parm7 → no AmberTools).'
        load_pdb([os.path.join(ex, 'complex.pdb')], parm=os.path.join(ex, 'complex.parm7'),
                 center=['SAM', 'PHN'], lcharge={'SAM': 1, 'PHN': -1},
                 scan_preset="[('SAM 359 CS1','PHN 360 C8',1.3)]")
ex_btn.on_click(_load_example)
_input_formats = ('.pdb / .cif / .mmcif / .xyz / .gjf' if IS_CLUSTER else
                  '.pdb / .cif / .mmcif + matching .parm7')
_drop = W.VBox([W.HTML('<div style="text-align:center;color:#567;"><b>Drag and drop</b> '
                       '%s here — or click to browse</div>' % _input_formats), upl])
_drop.add_class('rxdrop')
input_box = W.VBox([
    W.HTML('<b>Load a structure</b> — drop a file, or use the bundled example. '
           '<small>(Reactant first, product last for multi-PDB.)</small>'),
    _drop, W.HTML('<hr style="margin:6px 0">'), W.HBox([ex_choice, ex_btn]), input_msg])

# ============================================================== SELECT tab (3D pick)
viewer_out, ngl_out = W.Output(), W.Output()
viewer_status = W.HTML()
center_panel, charge_panel, scan_panel, freeze_panel, measure_panel = (W.VBox() for _ in range(5))
for _p in (center_panel, charge_panel, scan_panel, freeze_panel, measure_panel): _p.add_class('rxcard')
center_ids_html = W.HTML()
center_widget = None
charge_rows = None
pick_action = W.Dropdown(
    options=[('Center residue (-c)', 'center'), ('Ligand charge (-l)', 'ligand'),
             ('Scan bond · atom A', 'scanA'), ('Scan bond · atom B', 'scanB'),
             ('Freeze pair · atom A', 'freezeA'), ('Freeze pair · atom B', 'freezeB'),
             ('Freeze atom (Cartesian)', 'freezeatom'),
             ('Measure (dist/angle/dihedral)', 'measure')],
    value='center', description='Click sets:', style={'description_width': 'initial'},
    layout=W.Layout(width='360px', max_width='100%'))

_CLICK_JS = """function(atom,viewer,event,container){
 if(!atom)return;
 viewer.addStyle({index:atom.index},{stick:{color:'orange',radius:0.4}});
 viewer.addLabel((atom.chain||'')+':'+atom.resn+atom.resi+' '+atom.atom,{position:atom,backgroundColor:'black',fontColor:'white',fontSize:11});
 viewer.render();
 if(typeof google!=='undefined'&&google.colab){
  google.colab.kernel.invokeFunction('pdb2reaction_gui.on_click',
   [String(atom.index),atom.resn,String(atom.resi),atom.chain||'',atom.atom||'',String(atom.serial)],{});}}"""

_REP = {'cartoon': {'cartoon': {}}, 'sticks': {'stick': {}}, 'spheres': {'sphere': {}},
        'ball+stick': {'stick': {'radius': 0.15}, 'sphere': {'scale': 0.25}}, 'line': {'line': {}}}

def _base_style():
    sty = {k: dict(v) for k, v in _REP.get(S.get('rep', 'cartoon'), {'cartoon': {}}).items()}
    col = S.get('color', 'spectrum')
    for k in sty:
        if col == 'spectrum': sty[k]['color'] = 'spectrum'
        elif col == 'chain': sty[k]['colorscheme'] = 'chainHetatm'
        elif col == 'element': sty[k]['colorscheme'] = 'default'
    return sty

def _dash(v, p, q, label):
    if not p or not q: return
    v.addCylinder({'start': {'x': p[0], 'y': p[1], 'z': p[2]}, 'end': {'x': q[0], 'y': q[1], 'z': q[2]},
                   'radius': 0.04, 'dashed': True, 'fromCap': 1, 'toCap': 1, 'color': 'orange'})
    v.addLabel(label, {'position': {'x': (p[0]+q[0])/2, 'y': (p[1]+q[1])/2, 'z': (p[2]+q[2])/2},
                       'backgroundColor': 'white', 'fontColor': 'black', 'fontSize': 10})

def render_viewer():
    if not S.get('_pdb_text'):
        viewer_status.layout.display = ''
        viewer_status.value = (
            '<div role="status" style="min-height:180px;display:flex;align-items:center;'
            'justify-content:center;text-align:center;border:1px dashed #cbd5e1;'
            'border-radius:12px;background:#f8fafc;color:#64748b;padding:16px;">'
            '<div><b>No structure loaded</b><br><small>Load a PDB/mmCIF in the Input tab, '
            'then return here to pick residues and atoms.</small></div></div>')
        with viewer_out: clear_output()
        return
    viewer_status.value = ''
    viewer_status.layout.display = 'none'
    text = S['_pdb_text']; het = S.get('_hetero', [])
    with viewer_out:
        clear_output()
        v = py3Dmol.view(width='100%', height=440)
        v.addModel(text, 'pdb'); v.setStyle({}, _base_style())
        if not S['show_water']:
            v.setStyle({'resn': list(_WATER)}, {})
        if het: v.addStyle({'resn': het}, {'stick': {'colorscheme': 'greenCarbon'}})
        for rn in S.get('center', []):
            v.addStyle({'resn': rn}, {'stick': {'color': 'magenta', 'radius': 0.3}})
        for rid in S.get('center_ids', []):
            parts = rid.split(':')
            for meta in S.get('_atom_meta', []):
                same = False
                if len(parts) == 3:
                    meta_resi = str(meta.get('resseq')) + str(meta.get('icode') or '')
                    same = (str(meta.get('chain') or '') == parts[0] and
                            str(meta.get('resname') or '').upper() == parts[1].upper() and
                            meta_resi == parts[2])
                elif len(parts) == 1:
                    meta_resi = str(meta.get('resseq')) + str(meta.get('icode') or '')
                    same = meta_resi == parts[0]
                if same: v.addStyle({'index': meta['index']}, {'stick': {'color': 'magenta', 'radius': 0.3}})
        for s in S['scan_atoms']:
            if s: v.addStyle({'index': s.get('index')}, {'sphere': {'color': 'red', 'scale': 0.45}})
        for at in S.get('measure_atoms', []):
            v.addStyle({'index': at.get('index')}, {'sphere': {'color': 'blue', 'scale': 0.4}})
        sa, sb = S['scan_atoms']
        if sa and sb:
            d = scan_distance(); _dash(v, _xyz(sa), _xyz(sb), ('%.2f Å' % d) if d else 'scan')
        for p in S.get('freeze_pairs', []):
            pa, pb = _xyz(p['a']), _xyz(p['b'])
            if pa and pb: _dash(v, pa, pb, '%.2f Å' % math.dist(pa, pb))
        if S.get('surface'): v.addSurface(py3Dmol.VDW, {'opacity': 0.55, 'color': 'white'})
        v.setClickable({}, True, _CLICK_JS)
        v.setHoverable({}, True,
            "function(a,v){if(!a.hl){a.hl=v.addLabel(a.resn+a.resi+' '+a.atom,"
            "{position:a,backgroundColor:'black',fontColor:'white',fontSize:10});}}",
            "function(a,v){if(a.hl){v.removeLabel(a.hl);delete a.hl;}}")
        if S.get('spin'): v.spin(True)
        if S.get('_zoomsel') and (S.get('center') or S.get('center_ids')):
            zsel = {'resn': S['center']} if S['center'] else {}
            v.zoomTo(zsel if zsel else {})
        else:
            v.zoomTo()
        v.show()

def _render_center_ids():
    ids = S.get('center_ids', [])
    center_ids_html.value = (
        ('<small>+ exact residues: <code>%s</code></small>' % ','.join(ids)) if ids
        else ('<small>click any residue to add an exact <code>CHAIN:RESNAME:RESSEQ</code> center; '
              'the name list selects every matching residue</small>')
    )

def on_click(atom_index, resn='', resi='', chain='', atom='', serial=''):
    try: index = int(atom_index)
    except (TypeError, ValueError): index = -1
    if 0 <= index < len(S.get('_atom_meta', [])):
        meta = S['_atom_meta'][index]
        resn = str(meta.get('resname') or resn).strip()
        resi = str(meta.get('resseq') if meta.get('resseq') is not None else resi) + str(meta.get('icode') or '')
        chain = str(meta.get('chain') or chain).strip()
        atom = str(meta.get('name') or atom).strip()
        xyz = meta.get('xyz')
    else:
        resn = str(resn).strip(); resi = str(resi).strip()
        chain = str(chain).strip(); atom = str(atom).strip()
        xyz = S['_atoms'].get((str(chain or ''), resn, str(resi), atom))
    act = pick_action.value
    if act == 'center':
        rid = ('%s:%s:%s' % (chain, resn, resi)) if chain else str(resi)
        if rid not in S['center_ids']:
            S['center_ids'].append(rid); _render_center_ids()
    elif act == 'ligand' and charge_rows is not None and resn in charge_rows:
        charge_rows[resn]['use'].value = True
    elif act in ('scanA', 'scanB'):
        S['scan_atoms'][0 if act == 'scanA' else 1] = {
            'chain': chain, 'resn': resn, 'resi': resi, 'atom': atom, 'xyz': xyz, 'index': index}
        _render_scan_panel()
    elif act in ('freezeA', 'freezeB'):
        S['freeze_buf'][0 if act == 'freezeA' else 1] = {
            'chain': chain, 'resn': resn, 'resi': resi, 'atom': atom, 'xyz': xyz, 'index': index}
        _render_freeze_panel()
    elif act == 'freezeatom':
        idx = index + 1 if index >= 0 else None
        if idx is not None and idx not in S['freeze_atoms']:
            S['freeze_atoms'].append(idx); _render_freeze_panel()
    elif act == 'measure':
        S['measure_atoms'].append({'chain': chain, 'resn': resn, 'resi': resi,
                                   'atom': atom, 'xyz': xyz, 'index': index})
        if len(S['measure_atoms']) > 4: S['measure_atoms'].pop(0)
        _render_measure_panel(); render_viewer()
    refresh()
try:
    from google.colab import output as _co
    _co.register_callback('pdb2reaction_gui.on_click', on_click)
except Exception:
    _co = None              # click->Python only registers inside Google Colab

def _render_scan_panel():
    a, b = S['scan_atoms']
    def fmt(x): return _aspec(x) if x else '— click to pick —'
    d = scan_distance()
    dist_html = ('<b style="color:%s">current A–B: %.2f Å</b>' % (ACCENT, d)) if d is not None else \
                '<small>current A–B: (pick both atoms)</small>'
    tgt = W.BoundedFloatText(value=float(S['scan_target']), min=0.3, max=6.0, step=0.05,
                             description='target Å', layout=W.Layout(width='160px'))
    def _t(_): S['scan_target'] = tgt.value; refresh()
    tgt.observe(_t, names='value')
    def _addstage(_):
        if a and b: S['scan_stages'].append([{'a': a, 'b': b, 't': S['scan_target']}]); _render_scan_panel(); refresh()
    def _addconc(_):
        if a and b:
            if not S['scan_stages']: S['scan_stages'].append([])
            S['scan_stages'][-1].append({'a': a, 'b': b, 't': S['scan_target']}); _render_scan_panel(); refresh()
    def _clrstages(_): S['scan_stages'] = []; _render_scan_panel(); refresh()
    def _clrcur(_): S['scan_atoms'] = [None, None]; S['scan_preset'] = ''; _render_scan_panel(); render_viewer(); refresh()
    b_ns = W.Button(description='add stage', icon='plus', layout=W.Layout(width='120px'), tooltip='Add the current bond as a new sequential stage (staged scan).')
    b_cc = W.Button(description='add concerted', icon='plus', layout=W.Layout(width='135px'), tooltip='Add the current bond to the last stage (bonds changed together = concerted).')
    b_cs = W.Button(description='clear stages', layout=W.Layout(width='120px'))
    b_cl = W.Button(description='clear bond', icon='eraser', layout=W.Layout(width='110px'))
    b_ns.on_click(_addstage); b_cc.on_click(_addconc); b_cs.on_click(_clrstages); b_cl.on_click(_clrcur)
    summ = '<br>'.join('stage %d: %s' % (i + 1, ' & '.join('%s↔%s→%.3g Å' % (_aspec(x['a']), _aspec(x['b']), x['t']) for x in st))
                       for i, st in enumerate(S['scan_stages'])) or '<i>none (the single bond above is used)</i>'
    sub_now = _wv('dd_subcmd', S['subcmd'])
    if sub_now in ('scan2d', 'scan3d'):                          # 2D/3D grid: per-axis low/high ranges
        need = 2 if sub_now == 'scan2d' else 3
        lo = W.BoundedFloatText(value=1.2, min=0.3, max=6.0, step=0.05, description='low Å', layout=W.Layout(width='148px'))
        hi = W.BoundedFloatText(value=3.0, min=0.3, max=8.0, step=0.05, description='high Å', layout=W.Layout(width='148px'))
        def _addaxis(_):
            if a and b: S['scan_axes'].append({'a': a, 'b': b, 'lo': lo.value, 'hi': hi.value}); _render_scan_panel(); refresh()
        def _clraxes(_): S['scan_axes'] = []; _render_scan_panel(); refresh()
        b_ax = W.Button(description='add axis', icon='plus', layout=W.Layout(width='120px'),
                        tooltip='Add the current bond as a scan axis (low→high distance range).')
        b_cx = W.Button(description='clear axes', layout=W.Layout(width='110px'))
        b_ax.on_click(_addaxis); b_cx.on_click(_clraxes)
        axsum = '<br>'.join('axis %d: %s↔%s  [%.3g – %.3g Å]' % (k + 1, _aspec(x['a']), _aspec(x['b']), x['lo'], x['hi'])
                            for k, x in enumerate(S['scan_axes'])) or '<i>none</i>'
        col = '#1f7a3d' if len(S['scan_axes']) == need else '#a60'
        scan_panel.children = [
            W.HTML('<b>%s grid -s</b> <small>pick bond A/B, set low/high, “add axis” — need <b>%d</b> axes</small>' % (sub_now, need)),
            W.HTML('A: <code>%s</code><br>B: <code>%s</code><br>%s' % (fmt(a), fmt(b), dist_html)),
            W.HBox([lo, hi]), W.HBox([b_ax, b_cx]),
            W.HTML('<small style="color:%s"><b>axes (%d/%d):</b><br>%s</small>' % (col, len(S['scan_axes']), need, axsum))]
        return
    note = '' if len(S.get('inputs', [])) == 1 else \
        '<br><small style="color:#a60">(scan applies to single-PDB mode; ignored for multi-PDB MEP)</small>'
    scan_panel.children = [W.HTML('<b>Scan bond -s</b>%s' % note),
                           W.HTML('A: <code>%s</code><br>B: <code>%s</code><br>%s' % (fmt(a), fmt(b), dist_html)),
                           tgt, W.HBox([b_ns, b_cc]), W.HBox([b_cs, b_cl]),
                           W.HTML('<small><b>staged scan:</b><br>%s</small>' % summ)]

def _render_freeze_panel():
    fa, fb = S['freeze_buf']
    def fmt(x): return _aspec(x) if x else '— pick —'
    def _addpair(_):
        if fa and fb:
            S['freeze_pairs'].append({'a': fa, 'b': fb, 't': None}); S['freeze_buf'] = [None, None]
            _render_freeze_panel(); refresh()
    def _clrpairs(_): S['freeze_pairs'] = []; _render_freeze_panel(); refresh()
    def _clratoms(_): S['freeze_atoms'] = []; _render_freeze_panel(); refresh()
    b_ap = W.Button(description='add freeze pair', icon='plus', layout=W.Layout(width='160px'))
    b_cp = W.Button(description='clear pairs', layout=W.Layout(width='110px'))
    b_ca = W.Button(description='clear atoms', layout=W.Layout(width='110px'))
    b_ap.on_click(_addpair); b_cp.on_click(_clrpairs); b_ca.on_click(_clratoms)
    pairs = '; '.join('%s↔%s' % (_aspec(p['a']), _aspec(p['b'])) for p in S['freeze_pairs']) or '(none)'
    atoms = ','.join(str(i) for i in S['freeze_atoms']) or '(none)'
    freeze_panel.children = [
        W.HTML('<b>Distance restraints</b> <small>(<code>--dist-freeze</code>, opt) — set pick mode to '
               '“Freeze pair · A/B”, click two atoms, then add</small>'),
        W.HTML('A: <code>%s</code> &nbsp; B: <code>%s</code>' % (fmt(fa), fmt(fb))),
        W.HBox([b_ap, b_cp]),
        W.HTML('<small>pairs: <code>%s</code></small>' % pairs),
        W.HTML('<hr style="margin:5px 0"><b>Frozen atoms</b> <small>(<code>--freeze-atoms</code>, '
               'opt/tsopt/path-opt/freq/irc) — pick mode “Freeze atom”, click atoms</small>'),
        W.HBox([b_ca]),
        W.HTML('<small>1-based indices: <code>%s</code></small>' % atoms)]

def _render_measure_panel():
    pts = [_xyz(a) for a in S['measure_atoms']]
    names = ' , '.join(_aspec(a) for a in S['measure_atoms']) or '(none)'
    rows = []
    if len(pts) >= 2 and pts[0] and pts[1]:
        rows.append('distance(1–2): <b>%.3f Å</b>' % math.dist(pts[0], pts[1]))
    if len(pts) >= 3 and all(pts[:3]):
        ang = _angle(pts[0], pts[1], pts[2])
        if ang is not None: rows.append('angle(1-2-3): <b>%.1f°</b>' % ang)
    if len(pts) >= 4 and all(pts[:4]):
        rows.append('dihedral(1-2-3-4): <b>%.1f°</b>' % _dihedral(pts[0], pts[1], pts[2], pts[3]))
    clr = W.Button(description='clear measure', layout=W.Layout(width='130px'))
    clr.on_click(lambda _: (S.__setitem__('measure_atoms', []), _render_measure_panel(), render_viewer()))
    measure_panel.children = [
        W.HTML('<b>Measure</b> <small>(pick mode “Measure”; 2=dist · 3=angle · 4=dihedral)</small>'),
        W.HTML('<small>atoms: %s</small>' % names),
        W.HTML('<small>%s</small>' % ('<br>'.join(rows) or '—')), clr]

def build_selection():
    global center_widget, charge_rows
    if S['mode'] != 'pdb' or not S['inputs']:
        with viewer_out: clear_output(); print('Load a PDB first (Input tab).')
        return
    path = S['inputs'][0]
    _ngl_loaded['path'] = None
    try:
        text, metadata, viewer_path = _load_view_structure(path)
    except Exception as exc:
        with viewer_out: clear_output(); print('Could not prepare structure for the viewer:', exc)
        input_msg.value = '❌ structure load failed: <code>%s</code>' % exc
        return
    S['_pdb_path'] = viewer_path
    S['_pdb_text'] = text
    S['_atom_meta'] = metadata
    allr, het = parse_residues(text, metadata); S['_hetero'] = het
    S['_atoms'] = parse_atoms(text, metadata)
    initial_center = tuple(x for x in S.get('center', []) if x in allr)
    center_widget = W.SelectMultiple(options=allr, value=initial_center,
                                     rows=min(12, max(5, len(allr))), layout=W.Layout(width='190px'))
    charge_rows = {}
    rows = []
    for rn in het:
        pre = S.get('lcharge', {}).get(rn, 0)
        use = W.Checkbox(value=(rn in S.get('lcharge', {})), indent=False, layout=W.Layout(width='26px'))
        val = W.BoundedFloatText(value=float(pre), min=-9, max=9, step=1, layout=W.Layout(width='74px'))
        charge_rows[rn] = {'use': use, 'val': val}
        rows.append(W.HBox([use, W.Label(rn, layout=W.Layout(width='64px')), W.Label('q='), val]))
    def _sync(_=None):
        S['center'] = list(center_widget.value)
        S['lcharge'] = {r: x['val'].value for r, x in charge_rows.items() if x['use'].value}
        refresh()
    center_widget.observe(_sync, names='value')
    for x in charge_rows.values():
        x['use'].observe(_sync, names='value'); x['val'].observe(_sync, names='value')
    _render_center_ids()
    center_panel.children = [W.HTML('<b>center -c</b> <small>(name = all matches; 3D click = exact chain/residue)</small>'),
                             center_widget, center_ids_html]
    charge_panel.children = ([W.HTML('<b>ligand charges -l</b>')] +
                             (rows or [W.HTML('<i>no hetero ligands</i>')]))
    _render_scan_panel()
    _render_freeze_panel()
    _render_measure_panel()
    render_viewer()
    with ngl_out:
        clear_output(); print('(open this panel to load the NGLView viewer)')
    _sync()

# NGLView is heavy on big systems -> load lazily, only when its panel is opened.
_ngl_loaded = {'path': None}
def _on_ngl_open(change):
    if ngl_acc.selected_index != 0: return
    if not _HAVE_NGL:
        with ngl_out: clear_output(); print('NGLView not installed (py3Dmol above still works).'); return
    if S.get('_pdb_path') and _ngl_loaded['path'] != S['_pdb_path']:
        _ngl_loaded['path'] = S['_pdb_path']
        with ngl_out:
            clear_output()
            try:
                view = nv.show_file(S['_pdb_path'])
                view.add_representation('licorice', selection='hetero and not water')
                view.center(); display(view)
            except Exception as e:
                print('NGLView unavailable:', e)   # optional; py3Dmol above still works

# PyMOL-like view controls
dd_rep = W.Dropdown(options=['cartoon', 'sticks', 'ball+stick', 'spheres', 'line'], value='cartoon',
                    description='rep', style={'description_width': 'initial'}, layout=W.Layout(width='150px'))
dd_col = W.Dropdown(options=['spectrum', 'chain', 'element'], value='spectrum',
                    description='color', style={'description_width': 'initial'}, layout=W.Layout(width='150px'))
cb_water = W.Checkbox(value=False, description='water', indent=False,
                      layout=W.Layout(width='82px'))
cb_surf  = W.Checkbox(value=False, description='surface', indent=False,
                      layout=W.Layout(width='92px'))
cb_spin  = W.Checkbox(value=False, description='spin', indent=False,
                      layout=W.Layout(width='72px'))
btn_reset = W.Button(description='↺ render', icon='refresh', layout=W.Layout(width='110px'),
                     tooltip='Re-render the viewer (re-applies the current -c / scan / freeze highlights).')
btn_zoomsel = W.Button(description='zoom to -c', icon='search-plus', layout=W.Layout(width='120px'),
                       tooltip='Zoom to the current center selection.')
def _vopt(_=None):
    S['show_water'] = cb_water.value; S['surface'] = cb_surf.value; S['spin'] = cb_spin.value
    S['rep'] = dd_rep.value; S['color'] = dd_col.value; S['_zoomsel'] = False; render_viewer()
for c in (cb_water, cb_surf, cb_spin, dd_rep, dd_col): c.observe(_vopt, names='value')
btn_reset.on_click(lambda _: (S.__setitem__('_zoomsel', False), render_viewer()))
btn_zoomsel.on_click(lambda _: (S.__setitem__('_zoomsel', True), render_viewer()))
view_controls = W.HBox([dd_rep, dd_col, cb_water, cb_surf, cb_spin, btn_reset, btn_zoomsel])

ngl_acc = W.Accordion(children=[ngl_out]); ngl_acc.set_title(0, 'NGLView — optional rich viewer (open to load)')
ngl_acc.selected_index = None
ngl_acc.observe(_on_ngl_open, names='selected_index')
_sel_help = ('<b>Click residues/atoms in the 3D view to build the run.</b><br>'
             '<small>Pick what each click sets, then click in 3D. '
             'Every center click records the exact <code>CHAIN:RESNAME:RESSEQ</code>; '
             'the name checklist intentionally selects all copies with that residue name. '
             'You can also edit the panels or the command line directly.</small>')
summary_html = W.HTML()
def _render_summary():
    cw = center_widget
    cen = ','.join(list(cw.value) if cw is not None else S.get('center', [])) or '—'
    ids = ','.join(S.get('center_ids', [])) or '—'
    lc = ','.join('%s:%g' % (k, v) for k, v in S['lcharge'].items()) or '—'
    sc = ' ; '.join(scan_literals()) or '—'
    fp = '; '.join('%s↔%s' % (_aspec(p['a']), _aspec(p['b'])) for p in S['freeze_pairs']) or '—'
    fa = ','.join(str(i) for i in S['freeze_atoms']) or '—'
    summary_html.value = (
        '<div style="border:1px solid #cdd;border-radius:8px;padding:6px 9px;background:#f7fbff;">'
        '<b>Selection summary</b><br><small>'
        'inputs: <b>%d</b> · backend: <b>%s</b>/%s<br>'
        'center -c (all-by-name): <code>%s</code> &nbsp; (exact residues): <code>%s</code><br>'
        'charges -l: <code>%s</code><br>scan -s: <code>%s</code><br>'
        'freeze pairs: <code>%s</code> · freeze atoms: <code>%s</code></small></div>'
        % (len(S['inputs']), S.get('backend'), S.get('model'), cen, ids, lc, sc, fp, fa))

chips_box = W.HBox()
chips_box.add_class('rxchip')
def _render_chips():
    def mk(kind, key):
        def _rm(_):
            if kind == 'center' and center_widget is not None:
                center_widget.value = tuple(x for x in center_widget.value if x != key)
            elif kind == 'id':
                S['center_ids'] = [x for x in S['center_ids'] if x != key]
            elif kind == 'atom':
                S['freeze_atoms'] = [x for x in S['freeze_atoms'] if x != key]
            _render_chips(); refresh()
        return _rm
    btns = []
    cur = list(center_widget.value) if center_widget is not None else S.get('center', [])
    for rn in cur:
        b = W.Button(description='%s ✕' % rn, button_style='info', layout=W.Layout(width='auto'))
        b.on_click(mk('center', rn)); btns.append(b)
    for rid in S.get('center_ids', []):
        b = W.Button(description='%s ✕' % rid, button_style='warning', layout=W.Layout(width='auto'))
        b.on_click(mk('id', rid)); btns.append(b)
    for fa in S.get('freeze_atoms', []):
        b = W.Button(description='⚓%d ✕' % fa, layout=W.Layout(width='auto'))
        b.on_click(mk('atom', fa)); btns.append(b)
    chips_box.children = btns or [W.HTML('<small>no center/freeze selection yet — click in 3D</small>')]

sel_lang = W.Text(value='', placeholder='select by name, e.g.  resn SAM+GPP   (or just: SAM GPP MG)',
                  description='select -c', style={'description_width': 'initial'}, layout=W.Layout(width='70%'))
b_sel = W.Button(description='set center', icon='check', layout=W.Layout(width='120px'),
                 tooltip='Set the center (-c) to the named residues (PyMOL-style: resn NAME+NAME).')
def _apply_sel(_):
    if center_widget is None: return
    expr = sel_lang.value.replace('resn', ' ').replace('+', ' ').replace(',', ' ')
    want = {t.strip().upper() for t in expr.split() if t.strip()}
    sel = tuple(o for o in center_widget.options if o in want)
    if sel: center_widget.value = sel
    refresh()
b_sel.on_click(_apply_sel)

freeze_acc = W.Accordion(children=[W.VBox([freeze_panel, W.HTML('<hr style="margin:6px 0">'), measure_panel])])
freeze_acc.set_title(0, '⚙️ Freezing & measurement (advanced)')
freeze_acc.selected_index = None
viewer_out.layout = W.Layout(width='100%')
viewer_col = W.VBox([
    W.HTML('<b>3D workspace</b> <small>— choose what a click sets, then pick directly on the structure</small>'),
    pick_action, view_controls, viewer_status, viewer_out])
viewer_col.add_class('rxviewer')
selection_inspector = W.VBox([
    summary_html, chips_box,
    W.HTML('<small><b>Select all matches by residue name</b></small>'),
    sel_lang, b_sel])
selection_inspector.add_class('rxinspector')
workspace = W.HBox([viewer_col, selection_inspector])
workspace.add_class('rxworkspace')
select_box = W.VBox([
    W.HTML(_sel_help), workspace,
    W.HBox([center_panel, charge_panel, scan_panel]), freeze_acc, ngl_acc])

# ============================================================== OPTIONS tab
cb_advsub = W.Checkbox(value=False, description='show advanced subcommands (utilities)', indent=False)
dd_subcmd = W.Dropdown(options=BASIC_SUBS, value='all', description='subcommand',
                       style={'description_width': 'initial'}, layout=W.Layout(width='280px'))
subreq = W.HTML(); subcmd_note = W.HTML()
def _on_sub(_):
    sub = dd_subcmd.value; S['subcmd'] = sub
    subreq.value = '<small>📥 needs: %s</small>' % SUBREQ.get(sub, '(advanced — complete in the command line)')
    subcmd_note.value = ('' if (sub in COMPUTE or sub == 'extract') else
                         '<small style="color:#a60">utility subcommand — finish it in the command line below</small>')
    _rsp = globals().get('_render_scan_panel')
    if _rsp is not None: _rsp()                          # scan vs scan2d/3d builder
    refresh()
dd_subcmd.observe(_on_sub, names='value')
def _on_advsub(_):
    keep = dd_subcmd.value
    dd_subcmd.options = (SUBS if cb_advsub.value else BASIC_SUBS)
    dd_subcmd.value = keep if keep in dd_subcmd.options else 'all'
cb_advsub.observe(_on_advsub, names='value')
_on_sub(None)   # initialise the requirement hint for the default subcommand
all_mode = W.ToggleButtons(
    options=[('MEP (≥2 files)', 'mep'), ('Scan (1 file + bond)', 'scan'), ('TS-only (1 TS)', 'tsonly')],
    value='mep', description='all mode')
def _on_mode(_):
    if all_mode.value == 'tsonly': w_ts.value = True
    refresh()
all_mode.observe(_on_mode, names='value')

_bk0 = BACKEND if BACKEND in ('mace', 'uma') else 'mace'
dd_backend = W.Dropdown(options=['mace', 'uma'], value=_bk0,
                        description='-b backend', style={'description_width': 'initial'},
                        layout=W.Layout(width='170px'))
dd_model = W.Dropdown(options=MODELS[_bk0], value=S.get('model', DEFAULT_MODEL[_bk0]),
                      description='model', style={'description_width': 'initial'},
                      layout=W.Layout(width='230px'))
def _bk(_):
    S['backend'] = dd_backend.value
    dd_model.options = MODELS[dd_backend.value]
    dd_model.value = DEFAULT_MODEL[dd_backend.value]
    S['model'] = dd_model.value
    refresh()
def _mdl(_):
    S['model'] = dd_model.value; refresh()
dd_backend.observe(_bk, names='value'); dd_model.observe(_mdl, names='value')
w_ts = W.Checkbox(value=False, description='--tsopt (TS + IRC)', indent=False)
w_th = W.Checkbox(value=False, description='--thermo (freq + ΔG)', indent=False)
w_out = W.Text(value='result', description='out dir', layout=W.Layout(width='260px'))
w_reuse = W.Checkbox(value=False, description='reuse non-empty out dir', indent=False,
                     tooltip='Enable only to resume or intentionally add to an existing output directory.')
output_note = W.HTML()
w_q = W.IntText(value=0, description='-q (small mol)', style={'description_width': 'initial'},
                layout=W.Layout(width='220px'))
def _render_output_note():
    out = _effective_out_dir() if '_effective_out_dir' in globals() else (w_out.value or 'result')
    occupied = os.path.isdir(out) and bool(os.listdir(out))
    output_note.value = ('<small style="color:#a60">⚠ non-empty output exists; enable reuse to run into it</small>'
                         if occupied and not w_reuse.value else '')
def _sync_run(_=None):
    S['tsopt'] = w_ts.value; S['thermo'] = w_th.value
    S['out_dir'] = w_out.value or 'result'; S['charge'] = w_q.value; refresh()
def _sync_charge(_=None):
    S['charge'] = w_q.value; S['charge_explicit'] = True; refresh()
for w in (w_ts, w_th, w_out, w_reuse): w.observe(_sync_run, names='value')
w_q.observe(_sync_charge, names='value')

# advanced flags (gated to the relevant subcommands; “extra flags” is the universal catch-all)
adv_mult = W.IntText(value=1, description='-m mult', style={'description_width': 'initial'}, layout=W.Layout(width='160px'))
adv_prec = W.Dropdown(options=['auto', 'fp32', 'fp64'], value='auto', description='--precision', style={'description_width': 'initial'}, layout=W.Layout(width='190px'))
adv_det = W.Checkbox(value=False, description='--deterministic', indent=False)
adv_mep = W.Dropdown(options=['(default)', 'gsm', 'dmf'], value='(default)', description='--mep-mode', style={'description_width': 'initial'}, layout=W.Layout(width='200px'))
adv_thresh = W.Dropdown(options=['(default)', 'gau_loose', 'gau', 'gau_tight', 'gau_vtight', 'baker'], value='(default)', description='--thresh', style={'description_width': 'initial'}, layout=W.Layout(width='230px'))
adv_radius = W.FloatText(value=0.0, description='-r radius Å (0=skip)', style={'description_width': 'initial'}, layout=W.Layout(width='220px'))
adv_dft = W.Checkbox(value=False, description='--dft single-point', indent=False)
adv_dftfb = W.Text(value='', description='--dft-func-basis', placeholder='wb97x-d3,def2-tzvp', style={'description_width': 'initial'}, layout=W.Layout(width='330px'))
adv_extra = W.Text(value='', description='extra flags', placeholder='any other flags, e.g. --max-cycles 500 --opt-mode hess', style={'description_width': 'initial'}, layout=W.Layout(width='99%'))
for w in (adv_mult, adv_prec, adv_det, adv_mep, adv_thresh, adv_radius, adv_dft, adv_dftfb, adv_extra):
    w.observe(refresh, names='value')
adv_box = W.VBox([W.HBox([adv_mult, adv_prec, adv_det]),
                  W.HBox([adv_mep, adv_thresh, adv_radius]),
    W.HBox([adv_dft, adv_dftfb]), adv_extra,
    W.HTML('<small>Appended to the command (gated to the relevant subcommands). '
                         '“extra flags” is parsed with shell quoting and passed verbatim. The CLI rejects '
                         'unknown options; use Validate before a costly run.</small>')])
adv_acc = W.Accordion(children=[adv_box]); adv_acc.set_title(0, 'Advanced flags'); adv_acc.selected_index = None
options_box = W.VBox([
    W.HTML('<b>Subcommand</b> <small>(all = end-to-end; the basic list is fully GUI-driven — no CLI typing)</small>'),
    W.HBox([dd_subcmd, cb_advsub]), subreq, subcmd_note,
    W.HTML('<b>all workflow</b> <small>(MEP / scan / TS-only)</small>'), all_mode,
    W.HTML('<hr style="margin:6px 0">'),
    W.HTML('<b>MLIP backend &amp; model</b> <small>(MACE = no login; UMA needs HF login. '
           'Changing backend here does not install it: rerun Setup first. <code>auto</code> precision '
           'means UMA fp32 and MACE fp64.)</small>'),
    W.HBox([dd_backend, dd_model]),
    W.HTML('<b>Depth</b> (off = fast MEP only):'), W.HBox([w_ts, w_th]),
    W.HBox([w_q, w_out, w_reuse]), output_note,
    W.HTML('<hr style="margin:6px 0">'), adv_acc])

# ============================================================== RESULTS tab
res_out, traj_out, plot_out = W.Output(), W.Output(layout={'width': '400px'}), W.Output(layout={'width': '430px'})
frame_slider = W.IntSlider(min=0, max=0, value=0, description='frame', continuous_update=False,
                           readout=True, layout=W.Layout(width='560px'))
_TRAJ = {'frames': [], 'energies': []}

def _parse_trj(path):
    frames, energies = [], []
    lines = open(path).read().splitlines()
    i, n_lines = 0, len(lines)
    while i < n_lines:
        if not lines[i].strip():
            i += 1; continue
        try: n = int(lines[i].strip())
        except ValueError:
            i += 1; continue
        comment = lines[i + 1] if i + 1 < n_lines else ''
        frames.append('\n'.join(lines[i:i + 2 + n]))
        try: energies.append(float(comment.strip().split()[0]))
        except (ValueError, IndexError): energies.append(None)
        i += 2 + n
    return frames, energies

def _rel_kcal():
    es = _TRAJ['energies']
    base = es[0] if (es and es[0] is not None) else next((e for e in es if e is not None), None)
    if base is None: return None
    return [((e - base) * 627.509) if e is not None else None for e in es]

def _stationary(ys):
    """Stationary states from the profile: R (first), interior minima/TS (prominence>0.3), P (last)."""
    n = len(ys)
    if n < 2: return [(0, 'R')]
    pts = [(0, 'R')]
    for k in range(1, n - 1):
        a, b, c = ys[k - 1], ys[k], ys[k + 1]
        if not (a == a and b == b and c == c): continue                       # skip NaN
        if b > a and b >= c and (b - min(a, c)) > 0.3: pts.append((k, 'TS'))
        elif b < a and b <= c and (max(a, c) - b) > 0.3: pts.append((k, 'min'))
    pts.append((n - 1, 'P'))
    return pts

def _show_frame(i):
    fr = _TRAJ['frames']
    if not fr: return
    i = max(0, min(i, len(fr) - 1))
    with traj_out:
        clear_output()
        v = py3Dmol.view(width=380, height=320)
        v.addModel(fr[i], 'xyz'); v.setStyle({'stick': {}, 'sphere': {'scale': 0.3}})
        v.zoomTo(); v.show()
    with plot_out:
        clear_output()
        rk = _rel_kcal()
        if rk is None: print('(no per-frame energies in the trajectory)'); return
        import matplotlib.pyplot as plt
        plt.rcParams.update({'font.size': 10, 'figure.dpi': 120, 'savefig.dpi': 120,
                             'axes.edgecolor': '#cbd5e1', 'axes.linewidth': 0.9,
                             'xtick.color': '#64748b', 'ytick.color': '#64748b',
                             'axes.labelcolor': '#475569', 'text.color': '#33404d'})
        xs = list(range(len(rk))); ys = [y if y is not None else float('nan') for y in rk]
        stat = _stationary(ys)
        ni = min(stat, key=lambda s: abs(s[0] - i))[0] if stat else i        # nearest stationary state
        LINE, HALO, EMPH, MUTE = '#4C72B0', '#F4A259', '#E07B39', '#cfd6dd'
        fig, ax = plt.subplots(figsize=(4.7, 3.2))
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        ax.grid(True, axis='y', color='#edf0f4', lw=0.9, zorder=0)
        for (si, lab) in stat:                                               # horizontal state levels
            on = (si == ni)
            ax.axhline(ys[si], color=(EMPH if on else MUTE), lw=(2.4 if on else 1.0),
                       ls=('-' if on else (0, (4, 3))), alpha=(0.95 if on else 0.85), zorder=1)
            ax.text(xs[-1], ys[si], ' ' + lab, va='center', ha='left', fontsize=9,
                    color=('#b5601f' if on else '#9aa3ad'), fontweight=('bold' if on else 'normal'))
        ax.plot(xs, ys, '-', color=LINE, lw=2.0, zorder=2)
        ax.plot(xs, ys, 'o', color=LINE, ms=3.6, mec='white', mew=0.5, zorder=3)
        if ys[i] == ys[i]:                                                   # halo + solid centre (no harsh dot)
            ax.plot([i], [ys[i]], 'o', ms=22, color=HALO, alpha=0.30, mec='none', zorder=4)
            ax.plot([i], [ys[i]], 'o', ms=9.5, color=EMPH, mec='white', mew=1.0, zorder=5)
        ax.set_xlabel('image', fontsize=10); ax.set_ylabel('ΔE  (kcal/mol)', fontsize=10)
        ax.set_title('frame %d / %d   ·   %s   ·   ΔE = %.1f kcal/mol'
                     % (i, len(rk) - 1, dict(stat).get(ni, ''), ys[i] if ys[i] == ys[i] else 0.0),
                     fontsize=10, color='#33404d')
        ax.margins(x=0.10, y=0.16)
        fig.tight_layout(); plt.show(); plt.close(fig)

frame_slider.observe(lambda ch: _show_frame(frame_slider.value), names='value')

def _summary_html(out=None):
    out = out or _effective_out_dir()
    sj = os.path.join(out, 'summary.json')
    if not os.path.exists(sj): return ''
    d = json.load(open(sj))
    def val(x):
        try: return '%.1f' % float(x)
        except (TypeError, ValueError): return '—'
    post = {row.get('index'): row for row in (d.get('post_segments') or [])}
    rows = ''
    for k, seg in enumerate(d.get('segments', []) or []):
        idx = seg.get('index', k + 1); ps = post.get(idx, {})
        mlip = ps.get('mlip') if isinstance(ps.get('mlip'), dict) else {}
        gibbs = ps.get('gibbs_mlip') if isinstance(ps.get('gibbs_mlip'), dict) else {}
        refined = mlip.get('barrier_kcal') is not None
        barrier = mlip.get('barrier_kcal') if refined else seg.get('barrier_kcal')
        delta = mlip.get('delta_kcal') if refined else seg.get('delta_kcal')
        method = 'TSOPT+IRC' if refined else 'MEP band'
        imag = (ps.get('ts_imag') or {}).get('n_imag')
        rows += ('<tr><td>seg %02d</td><td>%s</td><td align=right>%s</td>'
                 '<td align=right>%s</td><td align=right>%s</td><td align=right>%s</td></tr>') % (
                     idx, method, val(barrier), val(gibbs.get('barrier_kcal')), val(delta),
                     '—' if imag is None else str(imag))
    rls = d.get('rate_limiting_step') or {}; rl = rls.get('barrier_kcal')
    h = ('<table style="border-collapse:collapse;" border=1 cellpadding=5>'
         '<tr><th>segment</th><th>source</th><th>ΔE‡</th><th>ΔG‡</th>'
         '<th>ΔE</th><th>n<sub>imag</sub></th></tr>%s</table>' % rows)
    if rl is not None:
        h += ('<br><b style="font-size:15px;">⚡ rate-limiting barrier: %s kcal/mol</b> '
              '<small>(%s)</small>' % (val(rl), rls.get('method') or 'method not recorded'))
    h += '<br><small>status: %s · images: %s · backend/model: %s / %s</small>' % (
        d.get('status'), d.get('n_images'), d.get('mlip_backend'), d.get('mlip_model'))
    return h

def _results(out=None):
    out = out or _effective_out_dir()
    with res_out:
        clear_output()
        for p in [os.path.join(out, 'energy_diagram_MEP.png')] + sorted(glob.glob(os.path.join(out, '*.png'))):
            if os.path.exists(p): display(Image(filename=p)); break
        else: print('No energy diagram in', out)
        sh = _summary_html(out)
        if sh: display(W.HTML(sh))
        import html as _h                                          # embed interactive PES (scan2d/3d) HTML
        pes = (glob.glob(os.path.join(out, '*landscape*.html')) + glob.glob(os.path.join(out, '*density*.html'))
               + sorted(glob.glob(os.path.join(out, '**', '*.html'), recursive=True)))
        if pes:
            try:
                display(W.HTML('<b>Potential-energy surface:</b> <code>%s</code>' % os.path.basename(pes[0])))
                display(HTML('<iframe srcdoc="%s" style="width:100%%;height:560px;border:1px solid #e6eaf0;'
                             'border-radius:10px;"></iframe>' % _h.escape(open(pes[0]).read(), quote=True)))
            except Exception as e:
                print('could not embed PES HTML:', e)
    cand = []
    for pattern in ('mep_trj.xyz', 'finished_irc_trj.xyz', '**/finished_irc_trj.xyz',
                    '**/irc_trj.xyz', '**/*trj*.xyz'):
        for path in glob.glob(os.path.join(out, pattern), recursive=True):
            if path not in cand: cand.append(path)
    if cand:
        _TRAJ['frames'], _TRAJ['energies'] = _parse_trj(cand[0])
        frame_slider.max = max(0, len(_TRAJ['frames']) - 1); frame_slider.value = 0
        _show_frame(0)
    else:
        with traj_out: clear_output(); print('(run first — no trajectory yet)')
        with plot_out: clear_output()
res_btn = W.Button(description='Show results', icon='bar-chart', layout=W.Layout(width='200px'))
res_btn.on_click(lambda _: _results(_effective_out_dir()))
dl_btn = W.Button(description='Download results (.zip)', icon='download', layout=W.Layout(width='250px'))
def _download(_):
    out = _effective_out_dir()
    if not os.path.isdir(out):
        with res_out: clear_output(); print('No results directory yet — run first.'); return
    z = shutil.make_archive(out + '_bundle', 'zip', out)
    try:
        from google.colab import files as _f; _f.download(z)
    except Exception as e:
        with res_out: print('Download needs Colab; zip saved at', z, '(', e, ')')
dl_btn.on_click(_download)
results_box = W.VBox([
    W.HBox([res_btn, dl_btn]), res_out,
    W.HTML('<hr style="margin:8px 0"><b>Structure ↔ energy</b> '
           '<small>— scrub the path; the red marker tracks the current frame</small>'),
    frame_slider, W.HBox([traj_out, plot_out])])

# ============================================================== command line + run (bottom, PyMOL-style)
logbox = W.Output(layout={'border': '1px solid #ccc', 'max_height': '380px', 'overflow': 'auto'})
def _stream(cmd):
    with logbox:
        clear_output(); print('$', ' '.join(shlex.quote(c) for c in cmd), '\n')
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout: print(line, end='')
        p.wait(); print('\n[exit %d]' % p.returncode)
    return p.returncode
def _argv():
    line = cmd_box.value.strip()
    if not line or line.startswith('#'):
        with logbox: clear_output(); print('No command yet — load an input (Input tab).')
        return None
    return shlex.split(line)       # execute exactly what the editable command line shows

def _effective_out_dir(argv=None):
    """Return the last output path in the editable command, with GUI fallback."""
    if argv is None:
        line = cmd_box.value.strip()
        try: argv = shlex.split(line) if line and not line.startswith('#') else []
        except ValueError: argv = []
    out = None
    flags = ('-o', '--out-dir', '--output')
    for i, token in enumerate(argv):
        if token in flags and i + 1 < len(argv):
            out = argv[i + 1]
        else:
            for flag_name in flags:
                if token.startswith(flag_name + '='):
                    out = token.split('=', 1)[1]
    return out or S.get('_last_out_dir') or S.get('out_dir') or 'result'
b_rebuild = W.Button(description='rebuild', icon='magic', layout=W.Layout(width='120px'),
                     tooltip='Regenerate the command from the current GUI selections.')
b_rebuild.on_click(lambda _: (_auto.__setitem__('on', True), refresh()))
b_copy = W.Button(description='copy', icon='copy', layout=W.Layout(width='100px'), tooltip='Copy the command line.')
def _copy(_):
    try:
        from google.colab import output as _o
        _o.eval_js('navigator.clipboard.writeText(%s)' % json.dumps(cmd_box.value))
        toast.value = '<small style="color:#1f7a3d">copied ✓</small>'
    except Exception as e:
        toast.value = '<small>copy needs Colab (%s)</small>' % e
b_copy.on_click(_copy)
b_clear = W.Button(description='clear selections', icon='eraser', layout=W.Layout(width='160px'))
def _clear_sel(_):
    if center_widget is not None: center_widget.value = ()
    if charge_rows is not None:
        for x in charge_rows.values(): x['use'].value = False
    S['center_ids'] = []; S['scan_atoms'] = [None, None]; S['scan_preset'] = ''
    _render_center_ids(); _render_scan_panel(); render_viewer(); refresh()
b_clear.on_click(_clear_sel)
b_validate = W.Button(description='Validate', button_style='info', icon='check', layout=W.Layout(width='130px'),
                      tooltip='Run the displayed compute command with --dry-run appended; no MLIP stage runs.')
b_run = W.Button(description='RUN exact command', button_style='danger', icon='play', layout=W.Layout(width='190px'),
                 tooltip='Execute exactly the command shown above. This may start a costly GPU calculation.')
def _set_running(value):
    for button in (b_validate, b_run, b_rebuild, b_clear): button.disabled = value
def _do_validate(_):
    _sync_run(); a = _argv()
    if not a: return
    sub = a[1] if len(a) > 1 else ''
    if sub not in COMPUTE:
        with logbox: clear_output(); print('Validate is available for compute subcommands; use the command help for this utility.')
        return
    if '--dry-run' not in a: a = a + ['--dry-run']
    _set_running(True)
    try:
        run_status.value = '<span style="background:#2563eb;color:#fff;padding:2px 9px;border-radius:11px;">🔎 validating…</span>'
        rc = _stream(a)
        run_status.value = ('<span style="background:#1f7a3d;color:#fff;padding:2px 9px;border-radius:11px;">✓ valid</span>'
                            if rc == 0 else
                            '<span style="background:#a00;color:#fff;padding:2px 9px;border-radius:11px;">✗ validation failed</span>')
    finally:
        _set_running(False)
b_validate.on_click(_do_validate)
def _do_run(_):
    _sync_run(); a = _argv()
    if not a: return
    out = _effective_out_dir(a)
    if os.path.isdir(out) and os.listdir(out) and not w_reuse.value:
        run_status.value = '<span style="background:#a00;color:#fff;padding:2px 9px;border-radius:11px;">✗ output exists</span>'
        with logbox:
            clear_output(); print('Refusing to mix runs in non-empty output directory:', out)
            print('Choose a new out dir, or explicitly enable "reuse non-empty out dir" in Options.')
        app.selected_index = 2
        return
    _set_running(True)
    try:
        run_status.value = '<span style="background:#b58900;color:#fff;padding:2px 9px;border-radius:11px;">⏳ running…</span>'
        rc = _stream(a)
        run_status.value = ('<span style="background:#1f7a3d;color:#fff;padding:2px 9px;border-radius:11px;">✓ done</span>'
                            if rc == 0 else
                            '<span style="background:#a00;color:#fff;padding:2px 9px;border-radius:11px;">✗ failed (exit %d)</span>' % rc)
        if rc == 0 and '--dry-run' not in a:
            S['_last_out_dir'] = out
            _results(out); app.selected_index = 3       # jump to Results after a real run
    finally:
        _set_running(False)
b_run.on_click(_do_run)
def _session_dict():
    if center_widget is not None: S['center'] = list(center_widget.value)
    if charge_rows is not None:
        S['lcharge'] = {r: x['val'].value for r, x in charge_rows.items() if x['use'].value}
    keys = ['tool', 'backend', 'model', 'subcmd', 'inputs', 'parm', 'mode', 'center', 'center_ids',
            'lcharge', 'scan_atoms', 'scan_stages', 'scan_axes', 'scan_target', 'scan_preset',
            'freeze_pairs', 'freeze_atoms', 'measure_atoms', 'charge', 'charge_explicit', 'tsopt', 'thermo',
            'out_dir', 'rep', 'color']
    d = {k: S.get(k) for k in keys}
    d['advanced'] = {'mult': _wv('adv_mult', 1), 'precision': _wv('adv_prec', 'auto'),
                     'deterministic': _wv('adv_det', False), 'mep_mode': _wv('adv_mep', '(default)'),
                     'thresh': _wv('adv_thresh', '(default)'), 'radius': _wv('adv_radius', 0.0),
                     'dft': _wv('adv_dft', False), 'dft_func_basis': _wv('adv_dftfb', ''),
                     'extra': _wv('adv_extra', '')}
    return d
def _save_session(_):
    with open('session.json', 'w') as fh: json.dump(_session_dict(), fh, indent=1)
    toast.value = '<small style="color:#1f7a3d">saved session.json</small>'
    try:
        from google.colab import files as _f; _f.download('session.json')
    except Exception:
        toast.value = '<small>saved session.json (working dir)</small>'   # not in Colab
def _apply_session(d):
    for k in ('backend', 'model', 'subcmd', 'out_dir', 'tsopt', 'thermo', 'charge',
              'rep', 'color', 'scan_target', 'scan_preset', 'parm', 'mode'):
        if k in d: S[k] = d[k]
    S['inputs'] = d.get('inputs', S['inputs'])
    S['center'] = d.get('center', []); S['center_ids'] = d.get('center_ids', [])
    S['lcharge'] = d.get('lcharge', {}); S['scan_atoms'] = d.get('scan_atoms', [None, None])
    S['scan_stages'] = d.get('scan_stages', []); S['scan_axes'] = d.get('scan_axes', [])
    S['freeze_pairs'] = d.get('freeze_pairs', []); S['freeze_atoms'] = d.get('freeze_atoms', [])
    S['measure_atoms'] = d.get('measure_atoms', [])
    S['charge_explicit'] = bool(d.get('charge_explicit', False))
    adv = d.get('advanced', {})
    if S['backend'] in dd_backend.options: dd_backend.value = S['backend']
    dd_model.options = MODELS.get(dd_backend.value, dd_model.options)
    if S.get('model') in dd_model.options: dd_model.value = S['model']
    if S.get('subcmd') in SUBS:
        if S['subcmd'] not in dd_subcmd.options: cb_advsub.value = True
        dd_subcmd.value = S['subcmd']
    w_ts.value = bool(S.get('tsopt')); w_th.value = bool(S.get('thermo'))
    w_out.value = S.get('out_dir', 'result')
    _saved_charge_explicit = S['charge_explicit']
    w_q.value = int(S.get('charge') or 0)
    S['charge_explicit'] = _saved_charge_explicit
    adv_mult.value = int(adv.get('mult', 1)); adv_prec.value = adv.get('precision', 'auto')
    adv_det.value = bool(adv.get('deterministic', False)); adv_mep.value = adv.get('mep_mode', '(default)')
    adv_thresh.value = adv.get('thresh', '(default)'); adv_radius.value = float(adv.get('radius', 0.0))
    adv_dft.value = bool(adv.get('dft', False)); adv_dftfb.value = adv.get('dft_func_basis', '')
    adv_extra.value = adv.get('extra', '')
    if S['mode'] == 'pdb' and S['inputs'] and os.path.exists(S['inputs'][0]):
        build_selection()
        if center_widget is not None:
            center_widget.value = tuple(o for o in center_widget.options if o in set(S['center']))
        if charge_rows is not None:
            for rn, x in charge_rows.items():
                x['use'].value = rn in S['lcharge']
                if rn in S['lcharge']: x['val'].value = float(S['lcharge'][rn])
    refresh()
up_sess = W.FileUpload(accept='.json', multiple=False, description='load session', layout=W.Layout(width='200px'))
def _on_load_sess(change):
    items = up_sess.value
    pairs = items.items() if isinstance(items, dict) else [(f['name'], f) for f in items]
    for name, meta in pairs:
        c = meta['content']
        try:
            _apply_session(json.loads(c.decode() if isinstance(c, (bytes, bytearray)) else c))
            toast.value = '<small style="color:#1f7a3d">loaded %s</small>' % name
        except Exception as e:
            toast.value = '<small style="color:#a00">load failed: %s</small>' % e
up_sess.observe(_on_load_sess, names='value')
b_save = W.Button(description='save session', icon='save', layout=W.Layout(width='150px')); b_save.on_click(_save_session)
session_row = W.HBox([W.HTML('<small>session:</small>'), b_save, up_sess])
utility_bar = W.HBox([b_rebuild, b_copy, b_clear])
primary_bar = W.HBox([b_validate, b_run]); primary_bar.add_class('rxrun')
run_bar = W.HBox([utility_bar, primary_bar],
                 layout=W.Layout(width='100%', justify_content='space-between'))
cmdline_box = W.VBox([
    W.HBox([W.HTML('<b>Command line</b> <small>— auto-built; Validate adds <code>--dry-run</code>; Run executes exactly this text</small>'),
            ready_chip, run_status, toast]),
    cmd_box, run_bar, session_row, logbox])

# ============================================================== assemble
app = W.Tab([input_box, select_box, options_box, results_box])
for i, t in enumerate(['1 Input', '2 Select', '3 Options', '4 Results']): app.set_title(i, t)
render_viewer()
refresh()
_t = 'pdb2reaction'
header = W.HTML(
    '<div style="background:#0f172a;border-radius:16px;padding:15px 20px;margin-bottom:8px;'
    'box-shadow:0 6px 20px rgba(15,23,42,0.18);">'
    '<div style="display:flex;align-items:center;flex-wrap:wrap;gap:10px;">'
    '<span style="font-size:21px;font-weight:700;letter-spacing:.2px;color:#f8fafc;">%s</span>'
    '<span style="background:#1e293b;color:#93c5fd;padding:3px 11px;border-radius:999px;'
    'font-size:11px;font-weight:600;letter-spacing:.3px;">REACTION-MECHANISM GUI</span></div>'
    '<div style="margin-top:6px;font-size:12.5px;color:#94a3b8;">backend '
    '<b style="color:#e2e8f0;">%s</b> · runs in <b style="color:#e2e8f0;">your own</b> Colab'
    '&nbsp;&nbsp;·&nbsp;&nbsp;<span style="color:#cbd5e1;">Input → Select in 3D → Validate → Run</span></div></div>'
    % (_t, BACKEND))
rootbox = W.VBox([header, app, W.HTML('<hr style="margin:10px 0">'), cmdline_box])
rootbox.add_class('rxapp')
display(rootbox)
